# Tutorial: Waveform and Gate Basics

Audience:
- Users who want a quick public example for `pysuqu.funclib.awgenerator` and `pysuqu.qubit.gate`.

Prerequisites:
- A Python environment with `numpy`, `matplotlib`, `qutip`, and `plotly` installed.
- Basic familiarity with IQ waveforms.

Learning goals:
- Build a pulse envelope and channel schedule.
- Import a synthetic complex waveform into the public schedule hierarchy.
- Run a minimal single-qubit gate simulation.


## Outline

1. Build a compact public pulse schedule.
2. Generate complex I/Q samples for inspection.
3. Import a synthetic waveform from arrays.
4. Run a minimal gate simulation.
5. Try a short parameter-variation exercise.


In [ ]:
from __future__ import annotations

import importlib.util
from contextlib import redirect_stdout
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np

required = ["numpy", "matplotlib", "qutip", "plotly"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(
        "Install the runtime dependencies first: " + ", ".join(missing)
    )

from pysuqu.funclib.awgenerator import (
    ChannelSchedule,
    EnvelopeParams,
    MixerParams,
    PulseEvent,
    WaveformGenerator,
    import_waveform,
)
from pysuqu.qubit.gate import SingleQubitGate


## Step 1 - Create a compact schedule

This schedule uses one cosine pulse and a simple mixer configuration.
The generated arrays are enough for quick inspection before moving to a solver.


In [ ]:
envelope = EnvelopeParams(
    name="x90",
    duration=8.0,
    peak_amp=0.2,
    shape_type="cosine",
    drag_coeff=0.05,
)
pulse = PulseEvent(
    start_time=4.0,
    envelope=envelope,
    name="x90",
    if_freq=0.05,
)
channel = ChannelSchedule(
    name="drive",
    sampling_rate=2.0,
    mixer_config=MixerParams(lo_freq=5.0),
    events=[pulse],
)

generator = WaveformGenerator(total_time=24.0, sample_rate=2.0)
waveform_complex, _ = generator.generate_channel_waveform(channel, return_complex=True)
t_axis = generator.t_axis

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_axis, np.real(waveform_complex), label="I")
ax.plot(t_axis, np.imag(waveform_complex), label="Q")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Amplitude (a.u.)")
ax.set_title("Generated complex control waveform")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

{
    "sample_count": int(len(t_axis)),
    "peak_amplitude": float(np.max(np.abs(waveform_complex))),
}


## Step 2 - Import a synthetic waveform from arrays

`import_waveform()` is useful when a schedule starts from arrays rather than symbolic envelopes.
Here we keep the waveform synthetic and fully public.


In [ ]:
custom_time = np.arange(0.0, 12.0, 0.5)
custom_wave = 0.08 * np.exp(1j * 2 * np.pi * 0.03 * custom_time) * np.hanning(custom_time.size)

imported_schedule = import_waveform(
    custom_time,
    custom_wave,
    target_type="schedule",
    name="synthetic_drag_preview",
    mixer_config=MixerParams(lo_freq=5.0),
)

imported_generator = WaveformGenerator(total_time=16.0, sample_rate=2.0)
imported_waveform, _ = imported_generator.generate_channel_waveform(
    imported_schedule,
    return_complex=True,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(imported_generator.t_axis, np.real(imported_waveform), label="Imported I")
ax.plot(imported_generator.t_axis, np.imag(imported_waveform), label="Imported Q")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Amplitude (a.u.)")
ax.set_title("Waveform re-imported into the public schedule type")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

{
    "event_count": len(imported_schedule.events),
    "shape_type": imported_schedule.events[0].envelope.shape_type,
    "schedule_name": imported_schedule.name,
}


## Step 3 - Run a minimal gate simulation

This keeps the pulse intentionally simple. The goal here is to show the public solver path,
not to optimize a calibrated gate.


In [ ]:
with redirect_stdout(StringIO()):
    gate = SingleQubitGate(
        total_time=24.0,
        sample_rate=2.0,
        qubit_frequency=5.0,
        qubit_anharmonicity=-0.25,
        energy_trunc_level=3,
    )

with redirect_stdout(StringIO()):
    gate.load_channel(channel, is_print=False)
    result = gate.run_simulation(channel=channel)

{
    "time_sample_count": len(result.times),
    "state_sample_count": len(result.states),
    "final_state_shape": tuple(result.states[-1].shape),
    "first_time_ns": float(result.times[0]),
    "last_time_ns": float(result.times[-1]),
}


## Exercises

- Change `drag_coeff` and compare the resulting I/Q traces.
- Replace the cosine envelope with `shape_type="gaussian"`.
- Common pitfall: forgetting that `SingleQubitGate` expects GHz-scale qubit inputs and ns-scale waveform timing.


In [ ]:
def waveform_peak_for_drag(drag_coeff: float) -> float:
    test_env = EnvelopeParams(
        name="scan",
        duration=8.0,
        peak_amp=0.2,
        shape_type="cosine",
        drag_coeff=drag_coeff,
    )
    test_channel = ChannelSchedule(
        name="drag_scan",
        sampling_rate=2.0,
        mixer_config=MixerParams(lo_freq=5.0),
        events=[PulseEvent(start_time=4.0, envelope=test_env, name="scan", if_freq=0.05)],
    )
    test_wave, _ = generator.generate_channel_waveform(test_channel, return_complex=True)
    return float(np.max(np.abs(test_wave)))

{drag: waveform_peak_for_drag(drag) for drag in (0.0, 0.05, 0.10)}
